# 01. Data Quality Assurance & Hygiene Check
**Project:** NovaHome International Market Entry Strategy 2026  
**Engagement Phase:** Phase 03 — Python Data Preparation & Exploratory Analysis  
**Author:** Senior Consulting Analyst  

### Business & Consulting Purpose
In tier-one management consulting, analytical integrity is non-negotiable. Before conducting market sizing or building financial models, consultants perform an exhaustive **Data Quality Audit**. 
"Garbage in, garbage out" (GIGO) is the leading cause of failed strategic expansions. If population baselines, currency conversions, or percentage bounds are corrupted, every downstream decision—from revenue projections to working capital requirements—will be fatally flawed.

This notebook executes:
1. Shape and schema inspection
2. Data type integrity verification
3. Missing value analysis
4. Duplicate records check
5. Domain boundary validation (percentage limits [0, 100] and positive costs)
6. Assertion-backed data hygiene certification


In [1]:
# Step 1: Import core analytics libraries
import os
import pandas as pd
import numpy as np

# Define relative file paths
RAW_DATA_PATH = os.path.join('..', 'data', 'raw', 'market_research.csv')
if not os.path.exists(RAW_DATA_PATH):
    # Fallback if executing from root
    RAW_DATA_PATH = os.path.join('data', 'raw', 'market_research.csv')

print(f"Loading raw market research data from: {RAW_DATA_PATH}")
df_raw = pd.read_csv(RAW_DATA_PATH)
print("Dataset successfully loaded into pandas DataFrame.")


Loading raw market research data from: data/raw/market_research.csv
Dataset successfully loaded into pandas DataFrame.


### Step 2: Shape, Dimensions, and Column Inspection
Understanding the dimensionality of the dataset:
* **Rows (Observations)**: Each row represents one candidate country.
* **Columns (Features/Indicators)**: Each column represents a macroeconomic, demographic, competitive, or cost metric.


In [2]:
# Inspect shape: (number of rows, number of columns)
num_rows, num_cols = df_raw.shape
print(f"Dataset Dimensions: {num_rows} rows (countries) x {num_cols} columns (indicators)")

# Display column names and non-null counts
print("
--- Column Schema & Data Types ---")
print(df_raw.info())


Dataset Dimensions: 10 rows (countries) x 14 columns (indicators)

--- Column Schema & Data Types ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   country                       10 non-null     object 
 1   population_m                  10 non-null     float64
 2   urban_population_pct          10 non-null     float64
 3   disposable_income_usd         10 non-null     int64  
 4   fitness_participation_pct     10 non-null     float64
 5   ecommerce_penetration_pct     10 non-null     float64
 6   home_fitness_demand_index     10 non-null     int64  
 7   market_growth_cagr_pct        10 non-null     float64
 8   avg_selling_price_usd         10 non-null     int64  
 9   import_logistics_cost_usd     10 non-null     int64  
 10  competitor_intensity_score    10 non-null     float64
 11  regulatory_complexity_sc

### Step 3: Missing Value Audit
Missing data (`NaN` or `None`) can distort aggregate statistics and break downstream financial models. We inspect the count and percentage of nulls for every column.


In [3]:
# Calculate missing values count and percentage
null_counts = df_raw.isnull().sum()
null_pct = (null_counts / len(df_raw)) * 100

missing_report = pd.DataFrame({
    'Missing_Count': null_counts,
    'Missing_Percentage': null_pct
})
print("--- Missing Value Report ---")
print(missing_report)

# Assertion: Verify zero missing values in raw dataset
assert df_raw.isnull().sum().sum() == 0, "DATA QUALITY FAILURE: Null values detected!"
print("
[PASS] Zero missing values detected across all columns.")


--- Missing Value Report ---
                             Missing_Count  Missing_Percentage
country                                  0                 0.0
population_m                             0                 0.0
urban_population_pct                     0                 0.0
disposable_income_usd                    0                 0.0
fitness_participation_pct                0                 0.0
ecommerce_penetration_pct                0                 0.0
home_fitness_demand_index                0                 0.0
market_growth_cagr_pct                   0                 0.0
avg_selling_price_usd                    0                 0.0
import_logistics_cost_usd                0                 0.0
competitor_intensity_score               0                 0.0
regulatory_complexity_score              0                 0.0
digital_ad_cost_index                    0                 0.0
data_status                              0                 0.0

[PASS] Zero missing value

### Step 4: Duplicate Country Records Inspection
In geographical market analysis, each candidate country must appear exactly once. Duplicate rows would double-count market share and corrupt sizing totals.


In [4]:
# Check for duplicate country entries
duplicates = df_raw.duplicated(subset=['country']).sum()
print(f"Duplicate country rows found: {duplicates}")

# Assertion: Zero duplicate countries allowed
assert duplicates == 0, "DATA QUALITY FAILURE: Duplicate country rows found!"
print("[PASS] Unique country constraint validated. Exactly 10 unique markets.")


Duplicate country rows found: 0
[PASS] Unique country constraint validated. Exactly 10 unique markets.


### Step 5: Domain Boundary & Percentage Validation
To ensure mathematical validity:
1. **Percentages** (`urban_population_pct`, `fitness_participation_pct`, `ecommerce_penetration_pct`) must strictly reside between **0.0% and 100.0%**.
2. **Economic Metrics** (`population_m`, `disposable_income_usd`, `avg_selling_price_usd`, `import_logistics_cost_usd`) must be strictly positive ($> 0$).
3. **Index Scores** (`competitor_intensity_score`, `regulatory_complexity_score`) must fall between **1.0 and 5.0**.


In [5]:
# Validate percentage columns
percentage_cols = ['urban_population_pct', 'fitness_participation_pct', 'ecommerce_penetration_pct']
for col in percentage_cols:
    invalid_pct = df_raw[(df_raw[col] < 0) | (df_raw[col] > 100)]
    assert len(invalid_pct) == 0, f"DATA QUALITY FAILURE: {col} has values outside [0, 100]!"
print("[PASS] All percentage columns are bounded within [0.0, 100.0]%.")

# Validate non-negative economic values
non_negative_cols = ['population_m', 'disposable_income_usd', 'avg_selling_price_usd', 'import_logistics_cost_usd', 'digital_ad_cost_index']
for col in non_negative_cols:
    invalid_neg = df_raw[df_raw[col] <= 0]
    assert len(invalid_neg) == 0, f"DATA QUALITY FAILURE: {col} has non-positive values!"
print("[PASS] All demographic and cost metrics are strictly positive.")

# Validate index bounds [1.0, 5.0]
index_cols = ['competitor_intensity_score', 'regulatory_complexity_score']
for col in index_cols:
    invalid_idx = df_raw[(df_raw[col] < 1.0) | (df_raw[col] > 5.0)]
    assert len(invalid_idx) == 0, f"DATA QUALITY FAILURE: {col} has values outside [1.0, 5.0]!"
print("[PASS] Competitor intensity and regulatory complexity scores reside in [1.0, 5.0].")


[PASS] All percentage columns are bounded within [0.0, 100.0]%.
[PASS] All demographic and cost metrics are strictly positive.
[PASS] Competitor intensity and regulatory complexity scores reside in [1.0, 5.0].


### Step 6: Provenance & Data Status Audit
Consulting integrity requires strict visibility into what data is empirically verified vs. estimated or synthetic.


In [6]:
# Check distribution of data_status
status_counts = df_raw['data_status'].value_counts()
print("--- Data Status Breakdown ---")
print(status_counts)

allowed_statuses = {'verified', 'estimated', 'proxy', 'synthetic_assumption'}
actual_statuses = set(df_raw['data_status'].unique())
assert actual_statuses.issubset(allowed_statuses), f"Invalid status detected: {actual_statuses - allowed_statuses}"
print(f"
[PASS] All rows strictly comply with research governance statuses: {actual_statuses}")


--- Data Status Breakdown ---
data_status
verified     7
estimated    2
proxy        1
Name: count, dtype: int64

[PASS] All rows strictly comply with research governance statuses: {'verified', 'estimated', 'proxy'}


### Summary of Data Quality Findings
* **Total Observations**: 10 Countries (8 International Candidates + 2 Domestic Benchmarks).
* **Completeness**: 100% (0 null values across 140 data points).
* **Uniqueness**: 100% (Zero duplicate countries).
* **Validity**: All percentages, indexes, and economic values reside within realistic theoretical bounds.
* **Readiness**: The raw dataset has passed all automated quality gates and is certified for data cleaning and transformation in `02_data_cleaning.ipynb`.
